#1. Leitura das tabelas da Silver persistida

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    when,
    lit,
    round
)

pokemon = spark.table(
    "mvp_data_engineering.silver.pokemon"
)

combats = spark.table(
    "mvp_data_engineering.silver.combats"
)

print("Pokémon:", pokemon.count())
print("Batalhas:", combats.count())

Pokémon: 800
Batalhas: 50000


#2. Construindo a gold.dim_pokemon

In [0]:
dim_pokemon = pokemon.select(
    "pokemon_id",
    "name",
    "type_1",
    "type_2",
    "hp",
    "attack",
    "defense",
    "sp_attack",
    "sp_defense",
    "speed",
    "generation",
    "is_legendary"
)

print("Registros:", dim_pokemon.count())

print(
    "Pokémon distintos:",
    dim_pokemon.select("pokemon_id").distinct().count()
)

Registros: 800
Pokémon distintos: 800


Essa tabela possui uma estrutura mais simples e, seguindo os princípios da arquitetura Medalhão, representa uma versão validada e preparada para consumo dos dados provenientes da camada Silver. Sua disponibilização na camada Gold garante que as análises realizadas pelos stakeholders utilizem exclusivamente dados tratados e modelados para consumo analítico, evitando o acesso direto às tabelas da camada Silver.

#3. Construindo a gold.facts_battles

In [0]:
fact_battles = combats.select(
    "battle_id",
    "first_pokemon_id",
    "second_pokemon_id",
    "winner_id",
    "loser_id",
    "first_pokemon_won"
)

display(fact_battles.limit(10))

print("Combates Silver:", combats.count())
print("Combates Gold:", fact_battles.count())

battle_id,first_pokemon_id,second_pokemon_id,winner_id,loser_id,first_pokemon_won
0,266,298,298,266,false
1,702,701,701,702,false
2,191,668,668,191,false
3,237,683,683,237,false
4,151,231,151,231,true
5,657,752,657,752,true
6,192,134,134,192,false
7,73,545,545,73,false
8,220,763,763,220,false
9,302,31,31,302,false


Combates Silver: 50000
Combates Gold: 50000


Uma linha da fact_battles representa uma batalha realizada entre dois Pokémon.

Essa tabela possui uma estrutura mais simples e, seguindo os princípios da arquitetura Medalhão, representa uma versão validada e preparada para consumo dos dados provenientes da camada Silver. Sua disponibilização na camada Gold garante que as análises realizadas pelos stakeholders utilizem exclusivamente dados tratados e modelados para consumo analítico, evitando o acesso direto às tabelas da camada Silver.

#4. Persistindo as tabelas na camada Gold.

Primeira ingestão para que possamos criar a tabela agregada diretamente na mesma camada.

In [0]:
(
    dim_pokemon.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_data_engineering.gold.dim_pokemon")
)

(
    fact_battles.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_data_engineering.gold.fact_battles")
)

In [0]:
%sql

SHOW TABLES IN mvp_data_engineering.gold;

database,tableName,isTemporary
gold,dim_pokemon,false
gold,fact_battles,false


#5. Construindo a tabela de desempenho

Etapa voltada a criação da tabela analítica "agg_pokemon_performance"

##5.1 Criando a relação de primeiro e segundo pokémon

Essencial para podermos somar as participações dos Pokémon, seja ele o primeiro ou o segundo na ordenação das batalhas.

In [0]:
#Participação do primeiro pokémon
first_participations = fact_battles.select(
    col("battle_id"),
    col("first_pokemon_id").alias("pokemon_id"),
    when(
        col("winner_id") == col("first_pokemon_id"),
        1
    ).otherwise(0).alias("win")
)

display(first_participations.limit(10))

battle_id,pokemon_id,win
0,266,0
1,702,0
2,191,0
3,237,0
4,151,1
5,657,1
6,192,0
7,73,0
8,220,0
9,302,0


In [0]:
#Participação do segundo pokémon
second_participations = fact_battles.select(
    col("battle_id"),
    col("second_pokemon_id").alias("pokemon_id"),
    when(
        col("winner_id") == col("second_pokemon_id"),
        1
    ).otherwise(0).alias("win")
)

In [0]:
#União das participação dos dois pokémon
battle_participations = (
    first_participations
    .unionByName(second_participations)
)

In [0]:
print("Batalhas:", fact_battles.count())

print(
    "Participações:",
    battle_participations.count()
)

print(
    "Esperado:",
    fact_battles.count() * 2
)

Batalhas: 50000
Participações: 100000
Esperado: 100000


Validação da agregação: se temos 50.000 registros de batalha, juntando as participações válidas dos primeiros e segundos pokémon, vamos ter o dobro de participações (100.000)

##5.2 Agregando por Pokémon e criando dados de performance

In [0]:
#Agregando por Pokémon
pokemon_performance = (
    battle_participations
    .groupBy("pokemon_id")
    .agg(
        count("*").alias("total_battles"),
        sum("win").alias("total_wins")
    )
)

In [0]:
#Criando derrotas
pokemon_performance = (
    pokemon_performance
    .withColumn(
        "total_losses",
        col("total_battles") - col("total_wins")
    )
)

In [0]:
#Criando taxa de vitória
pokemon_performance = (
    pokemon_performance
    .withColumn(
        "win_rate",
        round(
            col("total_wins") / col("total_battles"),
            4
        )
    )
)

In [0]:
display(
    pokemon_performance
    .orderBy(col("win_rate").desc())
    .limit(10)
)

pokemon_id,total_battles,total_wins,total_losses,win_rate
155,129,127,2,0.9845
513,119,116,3,0.9748
704,125,121,4,0.968
20,119,115,4,0.9664
154,141,136,5,0.9645
477,129,124,5,0.9612
727,127,122,5,0.9606
717,123,118,5,0.9593
165,125,119,6,0.952
350,120,114,6,0.95


#5.3 Unindo com a tabela "dim_pokemon" para inclusão de Pokémon que não batalharam

In [0]:
#Unindo com dim_pokemon
from pyspark.sql.functions import coalesce

agg_pokemon_performance = (
    dim_pokemon
    .select("pokemon_id")
    .join(
        pokemon_performance,
        on="pokemon_id",
        how="left"
    )
    .fillna({
        "total_battles": 0,
        "total_wins": 0,
        "total_losses": 0
    })
)

display(
    agg_pokemon_performance
    .orderBy("total_battles")
    .limit(10)
)

pokemon_id,total_battles,total_wins,total_losses,win_rate
236,0,0,0,null
90,0,0,0,null
12,0,0,0,null
419,0,0,0,null
183,0,0,0,null
782,0,0,0,null
322,0,0,0,null
479,0,0,0,null
78,0,0,0,null
46,0,0,0,null


In [0]:
print("Dimensão:", dim_pokemon.count())
print("Agregada:", agg_pokemon_performance.count())

Dimensão: 800
Agregada: 800


#5.4 Verificação de inconsistências

In [0]:
invalid_totals = agg_pokemon_performance.filter(
    col("total_battles")
    != col("total_wins") + col("total_losses")
)

print(
    "Registros com totais inconsistentes:",
    invalid_totals.count()
)

Registros com totais inconsistentes: 0


In [0]:
invalid_rates = agg_pokemon_performance.filter(
    (col("win_rate") < 0)
    | (col("win_rate") > 1)
)

print(
    "Taxas inválidas:",
    invalid_rates.count()
)

Taxas inválidas: 0


In [0]:
total_battles = fact_battles.count()

total_wins = (
    agg_pokemon_performance
    .agg(sum("total_wins"))
    .collect()[0][0]
)

print("Batalhas:", total_battles)
print("Vitórias:", total_wins)

Batalhas: 50000
Vitórias: 50000


In [0]:
total_participations = (
    agg_pokemon_performance
    .agg(sum("total_battles"))
    .collect()[0][0]
)

print("Participações:", total_participations)
print("Esperado:", total_battles * 2)

Participações: 100000
Esperado: 100000


#6. Persistindo a tabela agregada

In [0]:
(
    agg_pokemon_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "mvp_data_engineering.gold.agg_pokemon_performance"
    )
)

In [0]:
%sql

SHOW TABLES IN mvp_data_engineering.gold;

database,tableName,isTemporary
gold,agg_pokemon_performance,false
gold,dim_pokemon,false
gold,fact_battles,false


#7. Testando consultas na camada Gold.

In [0]:
%sql

SELECT
    p.name,
    p.type_1,
    p.speed,
    a.total_battles,
    a.total_wins,
    a.total_losses,
    ROUND(a.win_rate * 100, 2) AS win_rate_pct
FROM mvp_data_engineering.gold.agg_pokemon_performance a
INNER JOIN mvp_data_engineering.gold.dim_pokemon p
    ON a.pokemon_id = p.pokemon_id
WHERE a.total_battles >= 20
ORDER BY a.win_rate DESC
LIMIT 10;

name,type_1,speed,total_battles,total_wins,total_losses,win_rate_pct
Mega Aerodactyl,Rock,150,129,127,2,98.45
Weavile,Dark,125,119,116,3,97.48
Tornadus Therian Forme,Flying,121,125,121,4,96.8
Mega Beedrill,Bug,145,119,115,4,96.64
Aerodactyl,Rock,130,141,136,5,96.45
Mega Lopunny,Normal,135,129,124,5,96.12
Greninja,Water,122,127,122,5,96.06
Meloetta Pirouette Forme,Normal,128,123,118,5,95.93
Mega Mewtwo Y,Psychic,140,125,119,6,95.2
Mega Sharpedo,Water,105,120,114,6,95.0


In [0]:
%sql

SELECT
    p.is_legendary,
    COUNT(*) AS pokemon_count,
    ROUND(AVG(a.total_battles), 2) AS avg_battles,
    ROUND(AVG(a.win_rate) * 100, 2) AS avg_win_rate_pct
FROM mvp_data_engineering.gold.agg_pokemon_performance a
INNER JOIN mvp_data_engineering.gold.dim_pokemon p
    ON a.pokemon_id = p.pokemon_id
WHERE a.total_battles > 0
GROUP BY p.is_legendary
ORDER BY p.is_legendary;

is_legendary,pokemon_count,avg_battles,avg_win_rate_pct
false,720,127.52,47.62
true,64,127.94,77.91
